In [1]:
import pandas as pd
import requests
import io
import os
from datetime import datetime


# paths
save_dir = "/lakehouse/default/Files/data/raw/bse_trade_inc"
log_file = "/lakehouse/default/Files/data/raw/logs/pipeline_log.csv"

os.makedirs(save_dir, exist_ok=True)

# today's date
today = datetime.today()

date_str = today.strftime("%Y%m%d")
file_date = today.strftime("%d-%m-%Y")

file_path = os.path.join(
    save_dir,
    f"trade_{file_date}.csv"
)

try:

    url = (
        "https://www.bseindia.com/download/"
        "BhavCopy/Equity/"
        f"BhavCopy_BSE_CM_0_0_0_{date_str}_F_0000.CSV"
    )

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.bseindia.com/"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=60
    )

    if response.status_code != 200:
        raise Exception("No data yet")

    df = pd.read_csv(
        io.StringIO(response.text)
    )

    # keep BSE cash market stocks only
    df = df[
        (df["Sgmt"] == "CM") &
        (df["FinInstrmTp"] == "STK")
    ]

    # keep required columns
    df = df[[
        "TradDt",
        "TckrSymb",
        "OpnPric",
        "HghPric",
        "LwPric",
        "ClsPric",
        "TtlTradgVol"
    ]]

    # rename columns
    df.columns = [
        "Date",
        "Symbol",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]

    # save file
    df.to_csv(
        file_path,
        index=False
    )

    status = "SUCCESS"
    rows = len(df)
    message = "File saved"

except Exception as e:

    status = "FAILED"
    rows = 0
    message = str(e)


# log row
log_row = pd.DataFrame([{
    "timestamp": datetime.now(),
    "dataset": "bse_trade",
    "status": status,
    "rows": rows,
    "message": message
}])

if os.path.exists(log_file):

    old_log = pd.read_csv(log_file)

    log_df = pd.concat(
        [old_log, log_row],
        ignore_index=True
    )

else:
    log_df = log_row


log_df.to_csv(
    log_file,
    index=False
)

print(status)
print("Rows:", rows)
print(message)

StatementMeta(, e076d624-1291-47ea-9ebf-8ee61a48c7c3, 3, Finished, Available, Finished, False)

FAILED
Rows: 0
Error tokenizing data. C error: Expected 1 fields in line 7, saw 7

